In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import spacy
from pathlib import Path
from grammar_parser import Group1Parser, Group2Parser, Group3Parser, Group4Parser
from grammar_errors.error_checker import ErrorChecker

nlp = spacy.load('en_core_web_sm')
parsers = [
    Group1Parser(nlp, resolve=True),
    Group2Parser(nlp, resolve=True),
    Group3Parser(nlp, resolve=True),
    Group4Parser(nlp, resolve=True),
]
checker = ErrorChecker()

BASE = Path('../processed_data/sentences/Student-2')
L1_PATH = BASE / 'lesson-1' / 'validated-sentence-separation.txt'
L2_PATH = BASE / 'lesson-2' / 'validated-sentence-separation.txt'

SAMPLE = 0.15   # fraction of sentences used for this demo

def load_sentences(p):
    all_s = [s.strip() for s in p.read_text('utf-8').splitlines() if s.strip()]
    return all_s[:max(1, int(len(all_s) * SAMPLE))]

l1_sents = load_sentences(L1_PATH)
l2_sents = load_sentences(L2_PATH)
print(f'Lesson 1: {len(l1_sents)} sentences  |  Lesson 2: {len(l2_sents)} sentences  (demo {int(SAMPLE*100)}%)')

Lesson 1: 55 sentences  |  Lesson 2: 50 sentences  (demo 15%)


In [2]:
import json
from datetime import datetime

CACHE_DIR  = Path('../processed_data/cache')
CACHE_PATH = CACHE_DIR / 'grammar_comparison_student2.json'

def _cache_valid():
    if not CACHE_PATH.exists():
        return False
    c = json.loads(CACHE_PATH.read_text('utf-8'))
    return (c.get('l1_sample_size') == len(l1_sents) and
            c.get('l2_sample_size') == len(l2_sents))

if _cache_valid():
    print('Cache found — loading...')
    _c = json.loads(CACHE_PATH.read_text('utf-8'))
    g1 = _c['g1']
    g2 = _c['g2']
    e1 = _c['e1']
    e2 = _c['e2']
    print(f'  generated at: {_c["generated_at"]}')
else:
    print('No cache — computing grammar + errors...')

    # ── Grammar parsing ────────────────────────────────────────────────────────
    def parse_lesson(sentences):
        counts = {}
        for sent, doc in zip(sentences, nlp.pipe(sentences)):
            for parser in parsers:
                for m in parser.parse(doc):
                    sid = m['structure_id']
                    if sid not in counts:
                        counts[sid] = {
                            'structure_id':         m['structure_id'],
                            'category':              m['category'],
                            'guideword':             m['guideword'],
                            'lowest_level':          m['lowest_level'],
                            'lowest_level_numeric':  m['lowest_level_numeric'],
                            'count':                 0,
                            'example_sentence':      sent,
                            'example_span':          m['span_text'],
                        }
                    counts[sid]['count'] += 1
        return counts

    print('  Parsing lesson 1...')
    g1 = parse_lesson(l1_sents)
    print(f'    {len(g1)} structures')
    print('  Parsing lesson 2...')
    g2 = parse_lesson(l2_sents)
    print(f'    {len(g2)} structures')

    # ── Error analysis ─────────────────────────────────────────────────────────
    def analyse_errors(sentences):
        by_cat = {}
        for rec in checker.check_sentences(sentences):
            cat = rec['grammar_category']
            if cat not in by_cat:
                by_cat[cat] = {
                    'grammar_category':  cat,
                    'dimension_code':    rec['dimension_code'],
                    'dimension_label':   rec['dimension_label'],
                    'weight':            rec['weight'],
                    'count':             0,
                    'example_sentence':  rec['sentence'],
                    'example_matched':   rec['matched_text'],
                    'example_message':   rec['message'],
                    'example_offset':    rec['offset'],
                    'example_length':    rec['error_length'],
                }
            by_cat[cat]['count'] += 1
        return by_cat

    print('  Checking errors lesson 1...')
    e1 = analyse_errors(l1_sents)
    print(f'    {len(e1)} categories')
    print('  Checking errors lesson 2...')
    e2 = analyse_errors(l2_sents)
    print(f'    {len(e2)} categories')

    # ── Save cache ─────────────────────────────────────────────────────────────
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_PATH.write_text(json.dumps({
        'generated_at':   datetime.now().isoformat(),
        'l1_sample_size': len(l1_sents),
        'l2_sample_size': len(l2_sents),
        'g1': g1, 'g2': g2, 'e1': e1, 'e2': e2,
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'  Cache saved -> {CACHE_PATH}')

print('Done.')

No cache — computing grammar + errors...
  Parsing lesson 1...


    65 structures
  Parsing lesson 2...


    47 structures
  Checking errors lesson 1...


    5 categories
  Checking errors lesson 2...


    5 categories
  Cache saved -> ..\processed_data\cache\grammar_comparison_student2.json
Done.


In [3]:
# Derive aggregates from loaded g1/g2/e1/e2
all_sids = set(g1) | set(g2)
shared_sids = sorted([s for s in all_sids if s in g1 and s in g2],
                     key=lambda s: g1[s]['lowest_level_numeric'], reverse=True)
l1_only     = sorted([s for s in all_sids if s in g1 and s not in g2],
                     key=lambda s: g1[s]['lowest_level_numeric'], reverse=True)
l2_only     = sorted([s for s in all_sids if s in g2 and s not in g1],
                     key=lambda s: g2[s]['lowest_level_numeric'], reverse=True)

union_ordered = (shared_sids + l1_only + l2_only)[:8]

top_e1 = sorted(e1.values(), key=lambda x: x['count'], reverse=True)[:8]
top_e2 = sorted(e2.values(), key=lambda x: x['count'], reverse=True)[:8]
shared_cats = set(e1) & set(e2)

print(f'Structures — shared: {len(shared_sids)}, L1-only: {len(l1_only)}, L2-only: {len(l2_only)} → top {len(union_ordered)} shown')
print(f'Errors — L1: {len(e1)} categories, L2: {len(e2)} categories, shared: {len(shared_cats)}')

Structures — shared: 39, L1-only: 26, L2-only: 8 → top 8 shown
Errors — L1: 5 categories, L2: 5 categories, shared: 3


In [4]:
from IPython.display import HTML
import html as _html

# ── Color palettes ─────────────────────────────────────────────────────────────
CEFR_COLOR = {
    'A1': '#388e3c', 'A2': '#388e3c',
    'B1': '#1565c0', 'B2': '#1565c0',
    'C1': '#6a1b9a', 'C2': '#6a1b9a',
}
# Error cards: colored by severity (weight 1/2/3)
SEV_BG     = {1: '#F0FDF4', 2: '#FFFBEB', 3: '#FEF2F2'}
SEV_BORDER = {1: '#4ADE80', 2: '#FCD34D', 3: '#F87171'}
SEV_TEXT   = {1: '#166534', 2: '#78350F', 3: '#7F1D1D'}
SEV_LABEL  = {1: 'beginner', 2: 'medium', 3: 'high'}
# Error span highlight inside sentence — darker shade of severity border
SEV_SPAN_BG = {1: '#4ADE8066', 2: '#FCD34D88', 3: '#F87171AA'}

# Dimension badge (still shown inside each error card)
DIM_BORDER = {'A': '#3B82F6', 'B': '#F59E0B', 'C': '#10B981', 'D': '#8B5CF6'}

def cefr_badge(level):
    c = CEFR_COLOR.get(level, '#888')
    return ('<span style="background:' + c + ';color:#fff;border-radius:3px;'
            'padding:1px 6px;font-size:.72em;font-weight:700;margin-right:4px;flex-shrink:0">'
            + level + '</span>')

def cat_chip(cat):
    return ('<span style="background:#eceff1;color:#37474f;border-radius:3px;'
            'padding:1px 5px;font-size:.68em;margin-right:5px;flex-shrink:0">'
            + _html.escape(cat) + '</span>')

def dim_badge(code):
    c = DIM_BORDER.get(code, '#888')
    return ('<span style="background:' + c + ';color:#fff;border-radius:3px;'
            'padding:1px 6px;font-size:.7em;font-weight:700;margin-right:5px">'
            + code + '</span>')

def sev_badge(weight):
    c = SEV_TEXT.get(weight, '#555')
    bg = SEV_BG.get(weight, '#f5f5f5')
    label = SEV_LABEL.get(weight, str(weight))
    return ('<span style="background:' + bg + ';color:' + c + ';border:1px solid '
            + SEV_BORDER.get(weight, '#ccc') + ';border-radius:3px;'
            'padding:1px 6px;font-size:.7em;font-weight:600">' + label + '</span>')

def highlight_span(sentence, span_text):
    s   = _html.escape(sentence)
    t   = _html.escape(span_text)
    idx = s.lower().find(t.lower())
    if idx == -1:
        return '<em style="color:#555;font-size:.8em">' + s[:85] + ('…' if len(s)>85 else '') + '</em>'
    pre  = s[:idx]
    mid  = s[idx:idx+len(t)]
    post = s[idx+len(t):]
    full = (pre + '<b style="color:#1a237e;text-decoration:underline dotted">'
            + mid + '</b>' + post)
    if len(sentence) > 90:
        full = full[:90] + '…'
    return '<em style="color:#555;font-size:.8em">' + full + '</em>'

# ── Grammar structure row ──────────────────────────────────────────────────────
def struct_row(sid, meta_or_none, is_present):
    sid_esc = _html.escape(sid, quote=True)
    if not is_present:
        return (
            '<div class="struct-row absent" data-sid="' + sid_esc + '" '
            'onmouseenter="onStructHover(\'' + sid_esc + '\')" onmouseleave="onStructLeave()">'
            '<span style="color:#bbb;font-size:.8em">— not detected —</span>'
            '</div>'
        )
    m = meta_or_none
    return (
        '<div class="struct-row" data-sid="' + sid_esc + '" '
        'onmouseenter="onStructHover(\'' + sid_esc + '\')" onmouseleave="onStructLeave()">'
        '<div style="display:flex;align-items:center;margin-bottom:3px">'
        + cefr_badge(m['lowest_level'])
        + cat_chip(m['category'])
        + '<span class="guideword">' + _html.escape(m['guideword']) + '</span>'
        + '<span class="struct-count">' + str(m['count']) + '×</span>'
        '</div>'
        '<div style="padding-left:2px">'
        + highlight_span(m['example_sentence'], m['example_span'])
        + '</div>'
        '</div>'
    )

# ── Error card (colored by severity) ──────────────────────────────────────────
def error_card(e, is_shared):
    w      = e['weight']
    bg     = SEV_BG.get(w, '#f5f5f5')
    border = SEV_BORDER.get(w, '#ccc')
    span_bg = SEV_SPAN_BG.get(w, '#ccc')

    shared_badge = ''
    if is_shared:
        shared_badge = ('<span style="background:#ff6f00;color:#fff;border-radius:3px;'
                        'padding:1px 6px;font-size:.7em;margin-left:6px">shared</span>')

    sent = e['example_sentence']
    off  = e['example_offset']
    lng  = e['example_length']
    pre  = _html.escape(sent[:off])
    mid  = _html.escape(sent[off:off+lng])
    post = _html.escape(sent[off+lng:])
    err_span = ('<span style="background:' + span_bg + ';border-bottom:2px solid '
                + border + ';border-radius:2px;padding:0 2px">' + mid + '</span>')
    example_html = '<em style="font-size:.83em;color:#444">"' + pre + err_span + post + '"</em>'

    return (
        '<div style="background:' + bg + ';border:1px solid ' + border + ';border-radius:7px;'
        'padding:10px 14px;margin-bottom:10px">'
        '<div style="display:flex;align-items:center;margin-bottom:5px">'
        + dim_badge(e['dimension_code'])
        + '<strong style="font-size:.87em;flex:1">' + _html.escape(e['grammar_category']) + '</strong>'
        + shared_badge
        + '<span style="margin-left:8px">' + sev_badge(w) + '</span>'
        + '<span style="font-size:.8em;color:#555;margin-left:8px">' + str(e['count']) + '×</span>'
        '</div>'
        + example_html
        + '<div style="font-size:.73em;color:#777;margin-top:4px">'
        + _html.escape(e['example_message']) + '</div>'
        '</div>'
    )

# ── Assemble ───────────────────────────────────────────────────────────────────
rows_l1  = ''.join(struct_row(sid, g1.get(sid), sid in g1) for sid in union_ordered)
rows_l2  = ''.join(struct_row(sid, g2.get(sid), sid in g2) for sid in union_ordered)
cards_l1 = ''.join(error_card(e, e['grammar_category'] in shared_cats) for e in top_e1)
cards_l2 = ''.join(error_card(e, e['grammar_category'] in shared_cats) for e in top_e2)

def stat_pill(label, value, color='#37474f'):
    return ('<span style="background:#f5f5f5;border-radius:4px;padding:3px 10px;'
            'margin-right:8px;font-size:.83em"><b style="color:' + color + '">'
            + str(value) + '</b> ' + label + '</span>')

stats_l1 = (stat_pill('sentences', len(l1_sents))
          + stat_pill('structures', len(g1), '#1565c0')
          + stat_pill('error types', len(e1), '#c62828'))
stats_l2 = (stat_pill('sentences', len(l2_sents))
          + stat_pill('structures', len(g2), '#1565c0')
          + stat_pill('error types', len(e2), '#c62828'))

CSS = '''
<style>
  .cmp-root  { font-family:sans-serif; }
  .cmp-title { font-size:1.25em; font-weight:700; margin-bottom:10px; color:#263238; }
  .cmp-cols  { display:grid; grid-template-columns:1fr 1fr; gap:18px; }
  .cmp-col-head { font-size:1em; font-weight:700; margin-bottom:6px;
                  padding:6px 12px; border-radius:6px 6px 0 0; color:#fff; }
  .cmp-box   { border:1px solid #cfd8dc; border-radius:0 0 6px 6px;
               overflow-y:auto; padding:10px; background:#fff; }
  .box-top   { height:340px; }
  .box-bot   { height:400px; margin-top:14px; }
  .box-label { font-size:.76em; font-weight:600; color:#607d8b;
               letter-spacing:.04em; text-transform:uppercase; margin-bottom:5px; }
  .struct-row { padding:7px 8px; border-radius:5px; margin-bottom:5px;
                cursor:default; border:1px solid transparent; transition:background .12s; }
  .struct-row:not(.absent):hover { background:#e3f2fd; border-color:#90caf9; }
  .struct-row.absent { opacity:.32; border:1px dashed #ccc; }
  .struct-row.hovered { background:#fff9c4 !important; border-color:#f9a825 !important;
                        opacity:1 !important; }
  .guideword    { font-size:.85em; flex:1; color:#263238; }
  .struct-count { font-size:.76em; color:#888; margin-left:auto; padding-left:8px; flex-shrink:0; }
  .stats-bar    { margin-bottom:5px; }
</style>
'''

JS = '''
<script>
function onStructHover(sid) {
  document.querySelectorAll("[data-sid]").forEach(function(r){r.classList.remove("hovered");});
  document.querySelectorAll("[data-sid=\\""+sid+"\\"]").forEach(function(r){r.classList.add("hovered");});
}
function onStructLeave() {
  document.querySelectorAll("[data-sid]").forEach(function(r){r.classList.remove("hovered");});
}
(function(){
  var t1=document.getElementById("top-l1"),t2=document.getElementById("top-l2");
  if(!t1||!t2)return;
  var lk=false;
  t1.addEventListener("scroll",function(){if(lk)return;lk=true;t2.scrollTop=t1.scrollTop;lk=false;});
  t2.addEventListener("scroll",function(){if(lk)return;lk=true;t1.scrollTop=t2.scrollTop;lk=false;});
})();
</script>
'''

HTML_OUT = (
    CSS
    + '<div class="cmp-root">'
    + '<div class="cmp-title">Student-2 — Grammar Comparison: Lesson 1 vs Lesson 2</div>'
    + '<div class="cmp-cols">'

    + '<div>'
    + '<div class="cmp-col-head" style="background:#1565c0">Lesson 1</div>'
    + '<div class="stats-bar">' + stats_l1 + '</div>'
    + '<div class="box-label">Top grammar structures</div>'
    + '<div class="cmp-box box-top" id="top-l1">' + rows_l1 + '</div>'
    + '<div class="box-label" style="margin-top:12px">Most recurring errors</div>'
    + '<div class="cmp-box box-bot">' + cards_l1 + '</div>'
    + '</div>'

    + '<div>'
    + '<div class="cmp-col-head" style="background:#2e7d32">Lesson 2</div>'
    + '<div class="stats-bar">' + stats_l2 + '</div>'
    + '<div class="box-label">Top grammar structures</div>'
    + '<div class="cmp-box box-top" id="top-l2">' + rows_l2 + '</div>'
    + '<div class="box-label" style="margin-top:12px">Most recurring errors</div>'
    + '<div class="cmp-box box-bot">' + cards_l2 + '</div>'
    + '</div>'

    + '</div></div>' + JS
)

HTML(HTML_OUT)